### DESCRIPTION (inspired by Leetcode.com)
You're a logistics manager preparing to ship products from a warehouse. Each product type has both a quantity and a weight per item. Shipping boxes have TWO constraints:

Capacity limit: Maximum number of items per box
Weight limit: Maximum weight (kg) per box
Rules:

Each product type must be packed separately
All boxes have the same capacity and weight limits
A box can hold at most capacity items OR maxWeightPerBox kg, whichever comes first
Items must be whole numbers (can't pack fractional items)
Given arrays of quantities and weights per item, plus box and weight constraints, find the minimum box capacity needed to ship all products.

Example 1:

Input:

quantities = [8, 12, 5]
weights = [2, 3, 1]  # kg per item
maxBoxes = 6
maxWeightPerBox = 20  # kg
Output: 5

Explanation: With capacity 5:

Product 0 (8 items @ 2kg): min(5, 10, 8) = 5 items/box → needs 2 boxes (5 + 3 items)
Product 1 (12 items @ 3kg): min(5, 6, 12) = 5 items/box → needs 3 boxes (5 + 5 + 2 items)
Product 2 (5 items @ 1kg): min(5, 20, 5) = 5 items/box → needs 1 box Total: 6 boxes ≤ 6 ✓
Example 2:

Input:

quantities = [10, 15, 8]
weights = [5, 2, 3]
maxBoxes = 10
maxWeightPerBox = 15
Output: 4

Explanation: With capacity 4:

Product 0 (10 items @ 5kg): min(4, ⌊15/5⌋, remaining) = 3 items/box → needs 4 boxes
Product 1 (15 items @ 2kg): min(4, ⌊15/2⌋, remaining) = 4 items/box → needs 4 boxes
Product 2 (8 items @ 3kg): min(4, ⌊15/3⌋, remaining) = 4 items/box → needs 2 boxes Total: 10 boxes ≤ 10 ✓

In [ ]:
class Solution_V1:
    def minimumShippingCapacity(self, quantities: list[int], weights: list[int], maxBoxes: int, maxWeightPerBox: int) -> int:
        # 1. Calculate max capacity allowed for each product based on weight and product quantity
        # 2. Set left = the smallest value of max allowed capacity
        # and right = the largest value of the max allowed capacities
        # 3. Calculate mid from left and right
        # 4. check if using the mid capacity, boxes used are same or less than maxBoxes. 
        # If boxes exceed limit, set left to mid (need bigger capacity per box).

        max_allowed_capacity = []
        for q, w in zip(quantities, weights):
            max_allowed_capacity.append(min(q, maxWeightPerBox//w))
        print(max_allowed_capacity)
        left = min(max_allowed_capacity)
        right = max(max_allowed_capacity)
        
        while left <= right:
            mid = (left+right)//2
            print(f"left: {left}, right: {right}, mid: {mid}")
            boxes_needed = self._calculateBoxesNeeded(mid, quantities, max_allowed_capacity)         
            if boxes_needed > maxBoxes:
                left = mid+1
            else:
                right = mid-1

        return left

    def _calculateBoxesNeeded(self, target_capacity: int, quantities: list[int], max_allowed_capacity: list[int]) -> int:
        boxes_needed = 0

        for max_cap, qty in zip(max_allowed_capacity, quantities):
            cap = target_capacity if target_capacity <= max_cap else max_cap
            boxes_needed += (qty + cap - 1) // cap # calculate ceiling

        return boxes_needed
            

### Feedback
Your box-counting formula is correct, but the binary-search lower bound is too high. left = min(max_allowed_capacity) assumes the answer must be at least the smallest weight-limited capacity, which is false: capacity can be smaller and simply cause more boxes. For [20], weight allows 5 items, but capacity 4 uses exactly 5 boxes; for [4,4], capacity 2 uses 4 boxes. Search all valid capacities from 1 through max(quantities). Also handle impossible inputs where maxWeightPerBox // w == 0, since no box can hold even one item; otherwise division by zero can occur in _calculateBoxesNeeded. Your ceil(q / cap) calculation and monotonic binary search are otherwise efficient: O(n log max_quantity).

In [78]:
class Solution:
    def minimumShippingCapacity(self, quantities: list[int], weights: list[int], maxBoxes: int, maxWeightPerBox: int) -> int:
        # 1. Calculate max capacity allowed for each product based on weight and product quantity
        # 2. Set left = the smallest value of max allowed capacity
        # and right = the largest value of the max allowed capacities
        # 3. Calculate mid from left and right
        # 4. check if using the mid capacity, boxes used are same or less than maxBoxes. 
        # If boxes exceed limit, set left to mid (need bigger capacity per box).

        max_allowed_capacity = []
        for q, w in zip(quantities, weights):
            cap_by_weight = maxWeightPerBox//w
            if cap_by_weight == 0: 
                raise ValueError("Product weight exceeding max weight per box")
            max_allowed_capacity.append(min(q, cap_by_weight))
        print(max_allowed_capacity)
        left = 1
        right = max(max_allowed_capacity)
        
        while left <= right:
            mid = (left+right)//2
            boxes_needed = self._calculateBoxesNeeded(mid, quantities, max_allowed_capacity)         
            if boxes_needed > maxBoxes:
                left = mid+1
            else:
                right = mid-1

        return left

    def _calculateBoxesNeeded(self, target_capacity: int, quantities: list[int], max_allowed_capacity: list[int]) -> int:
        boxes_needed = 0

        for max_cap, qty in zip(max_allowed_capacity, quantities):
            cap = target_capacity if target_capacity <= max_cap else max_cap
            boxes_needed += (qty + cap - 1) // cap # calculate ceiling

        return boxes_needed
            

### Feedback

Your solution is correct and efficiently uses binary search: O(n log M) time and O(n) extra space, where M = max(quantities). Precomputing min(quantity, floor(weight limit / item weight)) is valid because no useful capacity exceeds either bound. A few interview-focused improvements:

Remove print(max_allowed_capacity); debug output should not remain in production/interview submissions.

ValueError is reasonable for an item too heavy for any box, but clarify the expected behavior if the problem guarantees valid inputs. Otherwise, returning an impossible indicator may be preferable.

Add a guard for empty quantities; max(...) currently raises an exception.

_calculateBoxesNeeded assumes cap > 0, which is safe under valid positive inputs but worth stating.

Your use of ceiling division is clean, and the monotonic feasibility condition is exactly what makes binary search appropriate.

In [1]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Input:
    quantities: list[int]
    weights: list[int]
    maxBoxes: int
    maxWeightPerBox: int
    
@dataclass(frozen=True)
class Test:
    input: Input
    expected_result: int

def run_tests(tests: list[Test], func: Callable[[list[int], list[int], int, int], int]):
    for test in tests:
        result = func(test.input.quantities, test.input.weights, test.input.maxBoxes, test.input.maxWeightPerBox)
        if result == test.expected_result:
            print("Test passed for " + str(test.input.quantities))
        else:
            print(f"Test failed for {test.input.quantities}. Expected: {test.expected_result}, Actual: {result}")

In [79]:
tests = [
    Test(Input([8, 12, 5], [2, 3, 1], 6, 20), 5),
    Test(Input([10, 15, 8], [5 ,2, 3], 10, 15), 4),
    Test(Input([20],[4],5,20),4),
    Test(Input([4,4], [1,1],4,10),2)
]

run_tests(tests, Solution().minimumShippingCapacity)

[8, 6, 5]
Test passed for [8, 12, 5]
[3, 7, 5]
Test passed for [10, 15, 8]
[5]
Test passed for [20]
[4, 4]
Test passed for [4, 4]
